[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# The Identity Map


## What you will be able to do

Say why two queries for the same row return one object, and use `Session.get` to find an object
without a query. Say what a commit forgets, and read an object's attributes back with `refresh`,
`expire` or a fresh query. Keep an object usable after its session closes, bring it back into a new
session with `merge`, and recognize the errors from a detached object, a stale one, a second object
for the same row, and a row that somebody else deleted.


## The idea

### The problem

The advising page shows a student's details at the top and the student's courses below, and the code
behind it loads Chloe Martin twice: once by email for the header, and once by id inside the list of
courses. If those were two objects, a change made through one, a corrected program at the top of the
page, would not show in the other, and the save at the end would have two versions of Chloe to
choose between.

After the save, the page reads Chloe's name once more to show it, and in a web application that
commits before it renders, that read raises an error about a session, although nothing about the
name changed. An application that keeps objects between requests, in a cache or behind a form, finds
them out of date, or refused outright by the session of the next request, because that session
already has a Chloe Martin of its own.

### What the identity map is

> The **identity map** is a session's record of the objects it has loaded, one for every row, keyed
> by the class and the primary key. A query whose rows include a row already in the map returns the
> object already there, so within one session, one row is always one object.
> **`session.get(Student, 3)`** looks in the map first, and asks the database only for a row the map
> does not hold. **Expiring** an object marks its attributes as out of date, so that the next read
> loads them again, and **`expire_on_commit`**, on unless the session is told otherwise, expires
> every object at every commit. An object whose session has closed is **detached**, and can load
> nothing. **`session.refresh()`** reloads an object at once, and **`session.merge()`** copies a
> detached object's values onto the session's own object for the same row.

### Why it works that way

- **One row, one object.** Two references to Chloe Martin in one session are the same object, so a
  change through one is a change through the other, and the flush has one version to write.
- **A query does not overwrite what the session holds.** When a row comes back for an object already
  in the map, the object keeps the values it has, so a change the program made and has not flushed is
  never lost to a query. The price is that a change made by another program is not seen until the
  object is expired or refreshed.
- **A commit forgets everything.** After a commit, another program may change any row, so the session
  expires every object, and the next read of an attribute runs a `SELECT` for it.
- **A detached object cannot load.** Its session is gone, so an expired attribute has nowhere to come
  from, and reading one raises `DetachedInstanceError`. Closing a session without a commit expires
  nothing, so the values loaded before the close stay readable.
- **An object belongs to one session at a time.** A detached object can be handed to a new session
  with `add()`, unless that session already holds an object for the same row. `merge()` is the answer
  then: it copies the detached object's values onto the session's own object and returns that.

### Where this shows up

A web application that commits a request's changes and then renders a page from the same objects is
where `DetachedInstanceError` is usually met, and `expire_on_commit=False` is the usual setting to
change because of it. The **Testing a Data Layer** notebook meets the same
error from a pytest fixture that returns an object after its session has closed. The identity map is
also why the **Declarative Models** notebook's dean's list could use students as dictionary keys:
every row came back as the same object, however many times it appeared.

### What this notebook covers

- Two queries, one object
- `session.get`, and the query it does not send
- What a commit forgets, and the `SELECT` that reads it back
- What another program's change looks like to a session
- Detached objects, and the values that survive a close
- Bringing a detached object back with `merge()`
- Which setting and which method to use when
- An edit form across three requests, finished
- Four errors, from an object read after its session closed to a row that somebody else deleted

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlalchemy import create_engine, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column


class Base(DeclarativeBase):
    pass


class Student(Base):
    __tablename__ = "students"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]


engine = create_engine("sqlite://")
Base.metadata.create_all(engine)
with Session(engine) as session:
    session.add(Student(id=3, name="Chloe Martin"))
    session.commit()
    by_query = session.scalars(select(Student).where(Student.name == "Chloe Martin")).one()
    by_key = session.get(Student, 3)
    print(by_query is by_key)
    by_key.name = "Chloe A. Martin"
    print(by_query.name)
```

```
True
Chloe A. Martin
```

A query by name and a lookup by key returned the same object, so the change made through one of
them is there when the other is read. Within a session, a row is one object, not a copy for every
query that finds it.


## Setup

Eleven imports, the college built from its classes, and three helpers.

- `sqlalchemy` is the library itself, and the cell prints its version
- `Session` and `sessionmaker`, from `sqlalchemy.orm`, make sessions, and `inspect`, from
  `sqlalchemy`, shows which attributes of an object have expired
- `DetachedInstanceError` and `ObjectDeletedError`, from `sqlalchemy.orm.exc`, and
  `InvalidRequestError`, from `sqlalchemy.exc`, are the errors Common errors catches
- `update` and `delete` change rows behind a session's back, as another program would, with
  `select`, `func`, `insert`, `create_engine` and `event`, and what the classes need
- `re` takes the memory address out of an error's message
- `StaticPool`, from `sqlalchemy.pool`, is the pool the helper uses for a database in memory
- `date` is what the `Date` columns take and return
- `logging` carries the SQL an engine logs to `PrintStatements`
- `Path` names the files, and `shutil` removes the scratch folder at the start and at the end

Setup builds the college from the classes of the **Declarative Models** notebook, as **The Session**
notebook did, with its `state` helper and a `SessionLocal`. `without_address` replaces the memory
address in an error's message, since a message such as `<Student at 0x104f7c050>` names a different
address on every run, so the errors in this notebook are printed through it.

Colab has SQLAlchemy installed, and this notebook runs version 2.0.54. Any 2.0 release runs it,
though an error may be worded a little differently. To match it exactly, run
`%pip install sqlalchemy==2.0.54` in a cell of its own, restart the session, and run this cell again.


In [1]:
import logging
import re
import shutil
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import (CheckConstraint, ForeignKey, MetaData, String, UniqueConstraint, create_engine, delete, event,
                        func, insert, inspect, select, update)
from sqlalchemy.exc import InvalidRequestError
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, sessionmaker
from sqlalchemy.orm.exc import DetachedInstanceError, ObjectDeletedError
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    @property
    def level(self):
        """100 for an introductory course, 200 for the next, read from the number in the code."""
        return int(self.code.split("-")[1]) // 100 * 100

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


def build_college(engine):
    """Create the college's tables from the classes, load the lists above into them, and count their rows."""
    Base.metadata.create_all(engine)
    rows = {
        Course: [{"code": code, "title": title, "department": department, "credits": credits}
                 for code, title, department, credits in COURSES],
        Student: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                  for name, email, program, started in STUDENTS],
        Term: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        Section: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        Enrollment: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                     for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for cls, values in rows.items():
            conn.execute(insert(cls), values)
        return {cls.__tablename__: conn.execute(select(func.count()).select_from(cls)).scalar_one() for cls in rows}


engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))


def state(obj):
    """Where an object stands with its session: transient, pending, persistent, deleted or detached."""
    info = inspect(obj)
    return next(name for name in ("transient", "pending", "persistent", "deleted", "detached") if getattr(info, name))


def without_address(error):
    """An error's message with every memory address replaced, since the addresses change on every run."""
    return re.sub(r"0x[0-9a-f]+", "0x...", str(error))


SessionLocal = sessionmaker(engine)


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}


## Worked examples

### Two queries, one object

Chloe Martin is found by email and by id in one session, and then again in a second session:


In [2]:
with SessionLocal() as session:
    by_email = session.scalars(select(Student).where(Student.email == "cmartin@college.edu")).one()
    by_id = session.scalars(select(Student).where(Student.id == 3)).one()
    print("one session, the same object:", by_email is by_id)
    print("its key in the identity map: ", inspect(by_id).identity_key)
    print("objects in the map:          ", len(session.identity_map))

with SessionLocal() as other:
    print("a second session's Chloe is the same object:", other.get(Student, 3) is by_id)


one session, the same object: True
its key in the identity map:  (<class '__main__.Student'>, (3,), None)
objects in the map:           1
a second session's Chloe is the same object: False


Two queries that found the same row returned one object, whose key in the map is the class and the
primary key, `(3,)`. The last part of the key is the identity token, for a session that spans several
databases, and is `None` here. The map belongs to the session, so a second session built an object of
its own for the same row: the identity map is a promise about one session, not about the program.

### session.get, and the query it does not send

`session.get` looks in the identity map before it asks the database. With `echo` on, the first lookup
sends a `SELECT`, and the second sends nothing:


In [3]:
engine.echo = True
with SessionLocal() as session:
    print("--- the first get")
    chloe = session.get(Student, 3)
    print("--- the second get")
    again = session.get(Student, 3)
    print("the same object:", again is chloe)
    print("--- a student who does not exist")
    print(session.get(Student, 99))
engine.echo = False


--- the first get
    BEGIN (implicit)
    SELECT students.id AS students_id, students.name AS students_name, students.email AS students_email, students.program AS students_program, students.started_on AS students_started_on
    FROM students
    WHERE students.id = ?
    values: (3,)
--- the second get
the same object: True
--- a student who does not exist
    SELECT students.id AS students_id, students.name AS students_name, students.email AS students_email, students.program AS students_program, students.started_on AS students_started_on
    FROM students
    WHERE students.id = ?
    values: (99,)
None
    ROLLBACK


The first `get` loaded Chloe Martin, and the second found the object in the map and returned it
without a word to the database. A missing student cost a query and returned `None`. A `select()`
always goes to the database, even for a row the map holds, and returns the object from the map once
the rows are back, so `session.get` is the way to look up by primary key.

### What a commit forgets, and the SELECT that reads it back

A commit expires every object the session holds. `inspect(obj).expired_attributes` lists what has
expired, and the next read of any of them loads the object again:


In [4]:
with SessionLocal() as session:
    chloe = session.get(Student, 3)
    print("expired before the commit:", sorted(inspect(chloe).expired_attributes))
    session.commit()
    print("expired after the commit: ", sorted(inspect(chloe).expired_attributes))

    engine.echo = True
    print("reading the name:", chloe.name)
    engine.echo = False
    print("expired after the read:   ", sorted(inspect(chloe).expired_attributes))


expired before the commit: []
expired after the commit:  ['email', 'id', 'name', 'program', 'started_on']
    BEGIN (implicit)
    SELECT students.id AS students_id, students.name AS students_name, students.email AS students_email, students.program AS students_program, students.started_on AS students_started_on
    FROM students
    WHERE students.id = ?
    values: (3,)
reading the name: Chloe Martin
expired after the read:    []


Nothing had expired before the commit, and every column had after it. Reading `chloe.name` began a
new transaction and loaded the whole row, which brought every attribute back, not just the name.
`session.expire(obj)` expires one object by hand, and `session.refresh(obj)` expires it and loads it
again at once.

### What another program's change looks like to a session

Expiring is how a session sees what other programs did. Here a second connection, standing in for
another program, moves Chloe Martin to History after the session has loaded Chloe and committed:


In [5]:
with SessionLocal() as session:
    chloe = session.get(Student, 3)
    print("loaded:           ", chloe.program)
    session.commit()                                        # expires Chloe

    with engine.begin() as conn:                            # another program changes Chloe's row
        conn.execute(update(Student).where(Student.id == 3).values(program="History"))

    print("after the change: ", chloe.program)


loaded:            Mathematics
after the change:  History


The session read the new program because the commit had expired Chloe, so the next read went to the
database. The other program could commit only because the session's own transaction had ended: while
a session holds a transaction that has read from SQLite, another program's commit waits for it, as
the **Connections and Transactions** notebook showed. Common errors shows what the same steps print
when nothing expires.

### Detached objects, and the values that survive a close

When a session closes, its objects are detached. A commit before the close had expired them, and a
close without a commit expires nothing:


In [6]:
with SessionLocal() as session:
    read_only = session.get(Student, 6)                     # a session that only reads, and closes

with SessionLocal() as session:
    committed = session.get(Student, 9)
    session.commit()                                        # expires Isabel Costa, then the session closes

print("read only:", state(read_only), "| expired:", sorted(inspect(read_only).expired_attributes), "|", read_only.name)
print("committed:", state(committed), "| expired:", sorted(inspect(committed).expired_attributes))


read only: detached | expired: [] | Felix Wagner
committed: detached | expired: ['email', 'id', 'name', 'program', 'started_on']


Both are detached. Felix Wagner, from a session that only read, keeps everything it loaded, and the
name reads without a database. Isabel Costa's session committed, so every attribute expired, and a
detached object has no session to load them from: reading Isabel's name is the first of the Common
errors. So a request that must commit and then use its objects either reads what it needs before the
commit, or makes its sessions with `expire_on_commit=False`, whose price Common errors also shows.

### Bringing a detached object back with merge()

A detached object can be given to a new session. `merge()` returns the new session's own object for
the same row, loaded from the database, with the detached object's values copied onto it:


In [7]:
read_only.program = "Mathematics"                           # a change made to the detached Felix Wagner

engine.echo = True
with SessionLocal.begin() as session:
    merged = session.merge(read_only)
    print("the same object:", merged is read_only, "| state:", state(merged), "| program:", merged.program)
engine.echo = False


    BEGIN (implicit)
    SELECT students.id AS students_id, students.name AS students_name, students.email AS students_email, students.program AS students_program, students.started_on AS students_started_on
    FROM students
    WHERE students.id = ?
    values: (6,)
the same object: False | state: persistent | program: Mathematics
    UPDATE students SET program=? WHERE students.id = ?
    values: ('Mathematics', 6)
    COMMIT


`merge()` loaded Felix Wagner's row into the session, copied the detached object's values onto the
session's object, and returned that object, not the detached one, which stays detached. The flush
found one change, the program, and sent it. `session.add(read_only)` would have attached the
detached object itself, which works only while the session holds no other object for that row.

### Which setting and which method to use when

| Use | When | Why |
|---|---|---|
| `session.get(Cls, key)` | a lookup by primary key | answered from the identity map when the object is there |
| `expire_on_commit=True`, the default | work that commits and goes on reading | every read after a commit sees the database as it is now |
| `expire_on_commit=False` | objects used after the commit that saved them, such as a page rendered from them | nothing expires, so nothing needs loading, at the price of stale values |
| `session.refresh(obj)` | one object that must be current now | expires it and loads it at once |
| `execution_options(populate_existing=True)` | a query whose rows must overwrite what the map holds | the fresh values replace the old ones |
| `session.merge(obj)` | a detached object going into a session that may hold the same row | copies its values onto the session's object |

The default is `expire_on_commit=True`, reading what a request needs before its commit, and
`session.get` for every lookup by key.

### An edit form across three requests, finished

The pieces of this notebook in three functions, one for every request of an edit form. The first
loads a student for the form with a session that only reads, so the object stays usable after the
close. The second saves the form's change with `merge()`, in a session that commits. The third builds
the page from plain values, read before its session closes:


In [8]:
def form_for(SessionLocal, student_id):
    """The first request: load a student for an edit form, and read nothing else."""
    with SessionLocal() as session:
        return session.get(Student, student_id)          # detached when the block ends, its columns still loaded


def save_form(SessionLocal, from_the_form, program):
    """The second request: save the form's change to the student it was made for."""
    with SessionLocal.begin() as session:
        student = session.merge(from_the_form)           # the session's own object, with the form's values on it
        student.program = program


def page_for(SessionLocal, student_id):
    """The third request: what the advising page shows, as plain values that outlive the session."""
    with SessionLocal() as session:
        student = session.get(Student, student_id)
        taking = session.scalar(select(func.count()).select_from(Enrollment).where(Enrollment.student_id == student_id,
                                                                                   Enrollment.status == "enrolled"))
        return {"name": student.name, "program": student.program, "courses this term": taking}



on_the_form = form_for(SessionLocal, 12)                    # Liam Murphy
print("the form shows:", on_the_form.name, "|", on_the_form.program, "|", state(on_the_form))

save_form(SessionLocal, on_the_form, "Psychology")
print("the page shows:", page_for(SessionLocal, 12))


the form shows: Liam Murphy | Computer Science | detached
the page shows: {'name': 'Liam Murphy', 'program': 'Psychology', 'courses this term': 3}


The form's object was detached from the start and still readable, since its session only read. The
save merged it into a session of its own, where the program changed from Computer Science to
Psychology, and the page read everything it shows while its session was open, so nothing it returns
can expire. `page_for` returns a dictionary rather than the `Student`, which is what makes the third
request safe whatever the session setting.

### Where each part came from

| In the three requests | What it relies on | The section that showed it |
|---|---|---|
| `session.get(Student, student_id)` | a lookup by key, answered from the map when it can be | session.get, and the query it does not send |
| `form_for` returning a detached object | a close without a commit expires nothing | Detached objects, and the values that survive a close |
| `session.merge(from_the_form)` | the session's own object, with the detached values copied on | Bringing a detached object back with merge() |
| `SessionLocal.begin()` in `save_form` | a commit at the end of the block | What a commit forgets, and the SELECT that reads it back |
| `page_for` returning a dictionary | values read before the session closed, which cannot expire | Detached objects, and the values that survive a close |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/10-the-identity-map-solutions.ipynb).

**1.** In one session, load all the History students with a query, then look up one of them with
`session.get`, and show that it is one of the objects the query returned.


In [9]:
# your code here


**2.** Load Daniel Kim, student 4, change Daniel's program without committing, run a query that
returns the same student again, and show that the query kept the change.


In [10]:
# your code here


**3.** Load a student, expire the object by hand with `session.expire`, print its expired
attributes, and read one of them with `echo` on.


In [11]:
# your code here


**4.** With a `sessionmaker` whose sessions use `expire_on_commit=False`, load a student, commit,
close the session, and read the name afterwards.


In [12]:
# your code here


**5.** Load a student in a session that only reads, change the email on the detached object, and save
the change with `merge()` in a session that commits.


In [13]:
# your code here


**6.** In one session, have another connection change a student's program after you have loaded that
student, then read the new program with `session.refresh`. Commit before the other connection
writes.


In [14]:
# your code here


## Common errors

The four errors below each name an object's memory address, which differs on every run, so every
cell catches its error and prints it through `without_address`.

### sqlalchemy.orm.exc.DetachedInstanceError: Instance <Student at 0x...> is not bound to a Session; attribute refresh operation cannot proceed


In [15]:
try:
    print(committed.name)                                   # Isabel Costa: expired by a commit, then detached
except DetachedInstanceError as error:
    print("DetachedInstanceError:", without_address(error))


DetachedInstanceError: Instance <Student at 0x...> is not bound to a Session; attribute refresh operation cannot proceed (Background on this error at: https://sqlalche.me/e/20/bhk3)


Isabel Costa's session committed, which expired the object, and then closed, which detached it.
Reading the name needed a `SELECT`, and a detached object has no session to run it. Read what the
program needs before the commit, as `page_for` does, or make the session with
`expire_on_commit=False`:


In [16]:
Keeping = sessionmaker(engine, expire_on_commit=False)

with Keeping() as session:
    kept = session.get(Student, 9)
    session.commit()

print(state(kept), "|", kept.name, "|", kept.program)


detached | Isabel Costa | Psychology


### No error, and yesterday's program: a stale read with expire_on_commit=False


In [17]:
with Keeping() as session:
    maya = session.get(Student, 13)                         # Maya Patel
    session.commit()                                        # expires nothing, with expire_on_commit=False

    with engine.begin() as conn:                            # another program changes Maya's row
        conn.execute(update(Student).where(Student.id == 13).values(program="History"))

    again = session.scalars(select(Student).where(Student.id == 13)).one()
    print("the query returned the same object:", again is maya, "| program:", maya.program)


the query returned the same object: True | program: Mathematics


The query went to the database and found History, and still returned Maya Patel's program as
Mathematics: the identity map returned the object it already held, and a query does not overwrite
what the session holds. With `expire_on_commit=False`, nothing had expired to make it look.
`refresh` loads one object again, and `populate_existing` makes a query overwrite what the map
holds:


In [18]:
ONLY_MAYA = select(Student).where(Student.id == 13)

with Keeping() as session:
    maya = session.get(Student, 13)                         # History now
    session.commit()

    with engine.begin() as conn:                            # the other program moves Maya again
        conn.execute(update(Student).where(Student.id == 13).values(program="Biology"))
    print("a plain query:      ", session.scalars(ONLY_MAYA).one().program)
    print("populate_existing:  ", session.scalars(ONLY_MAYA.execution_options(populate_existing=True)).one().program)
    session.commit()                                        # end the read, so the other program can write

    with engine.begin() as conn:
        conn.execute(update(Student).where(Student.id == 13).values(program="Mathematics"))
    session.refresh(maya)
    print("refresh:            ", maya.program)


a plain query:       History
populate_existing:   Biology
refresh:             Mathematics


A plain query returned what the session held, History, after the other program had written Biology.
`populate_existing` made the query overwrite the object with the row it found, and `refresh` loaded
the object again after the next change. The `commit()` in the middle ends the session's read, since
a session that has read from SQLite holds its transaction open, and another program's commit would
wait for it. A long-lived session with `expire_on_commit=False` has to ask for current values like
this, which is why it is not the default.

### sqlalchemy.exc.InvalidRequestError: Can't attach instance <Student at 0x...>; another instance with key (<class '__main__.Student'>, (5,), None) is already present in this session.


In [19]:
with SessionLocal() as session:
    kept_elena = session.get(Student, 5)                    # Elena Petrova, read and detached

kept_elena.program = "Psychology"
with SessionLocal() as session:
    loaded_elena = session.get(Student, 5)                  # this session has an Elena of its own
    try:
        session.add(kept_elena)
    except InvalidRequestError as error:
        print("InvalidRequestError:", without_address(error))


InvalidRequestError: Can't attach instance <Student at 0x...>; another instance with key (<class '__main__.Student'>, (5,), None) is already present in this session.


The session already held an object for Elena Petrova's row, and one row can be only one object, so
it refused a second. `add()` attaches the object it is given, and `merge()` copies it onto the one
already there:


In [20]:
with SessionLocal.begin() as session:
    loaded_elena = session.get(Student, 5)
    merged = session.merge(kept_elena)
    print("merge returned the session's own object:", merged is loaded_elena, "| program:", merged.program)


merge returned the session's own object: True | program: Psychology


### sqlalchemy.orm.exc.ObjectDeletedError: Instance '<Student at 0x...>' has been deleted, or its row is otherwise not present.


In [21]:
with SessionLocal() as session:
    yara = session.get(Student, 24)                         # Yara Haddad
    session.commit()                                        # expires Yara

    with engine.begin() as conn:                            # another program removes Yara and the enrollments
        conn.execute(delete(Enrollment).where(Enrollment.student_id == 24))
        conn.execute(delete(Student).where(Student.id == 24))

    try:
        print(yara.name)
    except ObjectDeletedError as error:
        print("ObjectDeletedError:", without_address(error))


ObjectDeletedError: Instance '<Student at 0x...>' has been deleted, or its row is otherwise not present.


Reading the expired name sent a `SELECT` for Yara Haddad's row, and the row was gone, deleted by
another program after the session loaded it. The object in memory could not be filled in, so the
session raised rather than return values that no longer exist. A program that may meet deleted rows
asks for the object again, and `session.get` answers `None` for a row that is not there:


In [22]:
with SessionLocal() as session:
    print("Yara Haddad now:", session.get(Student, 24))


Yara Haddad now: None


Last, the engine lets go of the file, and this cell removes the scratch folder, with the database in
it:


In [23]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- Within one session a row is one object: two queries that find it return the same object, and
  `session.get` answers from the identity map without a query.
- A query does not overwrite an object the session holds; a commit expires every object, so the next
  read loads the row again.
- `refresh`, `expire` and `populate_existing` load current values on purpose.
- A detached object keeps what it had loaded, and cannot load more: read before the commit, or use
  `expire_on_commit=False` and accept stale values.
- `merge()` brings a detached object into a session that may already hold its row, returning the
  session's own object.


## What is next

The **Relationships** notebook connects the classes: `relationship()` and `back_populates`, a
student's enrollments as a list, and the join condition it cannot determine.


---

&#8592; **Previous:** [The Session](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/09-the-session.ipynb)  &nbsp;·&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Relationships](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/11-relationships.ipynb) &#8594;
